# SMART EDUCATION ERP - HỆ THỐNG XÁC THỰC & PHÂN QUYỀN VAI TRÒ (RBAC)
## Tài liệu kỹ thuật giai đoạn thiết kế bảo mật Zero-Trust

## 1. Tổng quan kiến trúc xác thực (Authentication)
Hệ thống sử dụng **Firebase Authentication** kết hợp đồng bộ hóa hồ sơ người dùng trong Firestore nhằm đảm bảo an toàn thông tin theo nguyên tắc phân quyền tối thiểu (Least Privilege) và bảo mật Zero-Trust.

### Quy trình đăng nhập & Đồng bộ hóa hồ sơ:
1. **Xác thực danh tính**: Client gửi thông tin đăng nhập (Email/Password) trực tiếp đến máy chủ Firebase Auth.
2. **Kiểm soát trạng thái**: Sử dụng trình lắng nghe sự kiện `onAuthStateChanged` để theo dõi và quản lý phiên làm việc một cách phản ứng (reactive).
3. **Tải hồ sơ phân quyền**: Khi người dùng đăng nhập thành công, hệ thống truy vấn tài liệu trong bộ sưu tập `/users/{uid}` ở Firestore để xác định vai trò (`role`) và thông tin liên kết.
4. **Kiểm tra Zero-Trust**: Toàn bộ dữ liệu vai trò do máy chủ Firestore phản hồi từ thuộc tính hồ sơ thực tế thay vì tin tưởng vào dữ liệu do Client gửi lên.

## 2. Ma trận phân quyền hệ thống (Role-Based Access Control - RBAC)
Quyền truy cập các tab chức năng trong hệ thống được định nghĩa chặt chẽ thông qua hằng số `ALLOWED_TABS_BY_ROLE`:

| Vai trò (Role) | Mô tả | Danh sách Tab chức năng được phép truy cập | Quy tắc hạn chế | 
| :--- | :--- | :--- | :--- |
| **ADMIN** / **OWNER** | Chủ cơ sở / Quản trị viên cao cấp | Toàn bộ các tab chức năng | Không giới hạn | 
| **ACADEMIC_STAFF** | Nhân viên học vụ, giáo vụ | `dashboard`, `classes`, `students`, `homework`, `notifications`, `settings` | Không xem tài chính, không đổi quyền ADMIN | 
| **ACCOUNTANT** | Kế toán viên | `dashboard`, `tuition`, `notifications`, `settings` | Chỉ thao tác tài chính, thu phí | 
| **TEACHER** | Giáo viên đứng lớp | `dashboard`, `classes`, `students`, `homework`, `notifications`, `settings` | Chỉ thấy lớp học và bài tập của mình | 
| **STUDENT** | Học sinh | `dashboard`, `homework`, `notifications`, `settings` | Chỉ xem điểm cá nhân & làm bài tập | 
| **PARENT** | Phụ huynh | `dashboard`, `tuition`, `notifications`, `settings` | Theo dõi hóa đơn học phí và điểm con em |

## 3. Quy tắc bảo mật dữ liệu đầu cuối (Firestore Rules Security)
Các quy tắc bảo mật được triển khai trực tiếp thông qua tệp cấu hình `firestore.rules`. Hệ thống nghiêm cấm các quyền đọc/ghi tự do không xác thực (`if true;`).

### Ví dụ quy tắc bảo mật phân quyền:
```javascript
rules_version = '2';
service cloud.firestore {
  match /databases/{database}/documents {
    
    // Hàm kiểm tra danh tính chung
    function isAuthenticated() {
      return request.auth != null;
    }
    
    // Hàm kiểm tra vai trò cụ thể
    function getUserRole() {
      return get(/databases/$(database)/documents/users/$(request.auth.uid)).data.role;
    }

    match /users/{userId} {
      allow read: if isAuthenticated();
      allow write: if isAuthenticated() && (getUserRole() == 'ADMIN' || getUserRole() == 'OWNER');
    }
    
    match /auditLogs/{logId} {
      allow read, write: if isAuthenticated() && (getUserRole() == 'ADMIN' || getUserRole() == 'OWNER');
    }
  }
}
```

## 4. Nhật ký kiểm toán bảo mật tự động (Audit Logging)
Mọi hành động nhạy cảm của người dùng (Tạo lớp học, ghi danh, giao bài tập, tạo tài khoản, đăng xuất) đều được theo dõi chặt chẽ thông qua hàm tiện ích `logAuditEvent` để lưu vết trực tiếp vào phân hệ kiểm toán (`auditLogs`):

### Cấu trúc một bản ghi kiểm toán:
```typescript
interface AuditLog {
  id: string;        // Mã định danh tự sinh dạng duy nhất
  timestamp: string; // Thời điểm thực thi hành động chuẩn ISO-8601
  actor: string;     // Tên tài khoản thực thi hành động
  actorId: string;   // UID tài khoản thực thi
  action: string;    // Hành động nhạy cảm được thực thi
  target: string;    // Đối tượng chịu tác động (Mã lớp, mã học sinh...)
  status: string;    // Kết quả: 'Success' hoặc 'Failed'
  details: string;   // Mô tả chi tiết hành động kèm tham số
}
```